## 🚨 Crisis Detection & Early Warning System

This section helps identify critical groundwater conditions.
It highlights high-risk zones and generates early warning signals.

##### Import Libraries

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

##### Load Dataset

In [2]:
df = pd.read_csv("groundwater_ml_dataset_cleaned.csv")
df.head()

,state,district,annual_recharge,extractable_resource,annual_extraction,stage_of_development,category,extraction_ratio,utilization_rate,stress_level,risk_score,year
0,Himachal Pradesh,Himachal Pradesh,0.61,0.18,0.13,0.20,Safe,0.213115,0.722222,0.0020,0.374535,2024
1,Madhya Pradesh,Madhya Pradesh,27.00,1.68,0.17,7.04,Safe,0.006296,0.101190,0.0704,0.057075,2024
2,Andhra Pradesh,Alluri Sitharama Raju,43956.31,102516.48,2860.84,8890.60,Over Exploited,0.065084,0.027906,88.9060,17.818396,2024
3,Andhra Pradesh,Anakapalli,22443.69,38195.76,14423.44,6889.74,Over Exploited,0.642650,0.377619,68.8974,14.187588,2024
4,Andhra Pradesh,Ananthapuramu,40986.63,44150.14,1323.64,35512.65,Over Exploited,0.032294,0.029980,355.1265,71.050210,2024


### 🎨 Color Theme Used

- Soft alert-style colors  
- Gradient for risk intensity  
- Clean dashboard look  

### 1. Top Crisis Districts 
- Description

This chart displays the top 10 high-risk districts within each state. Users can dynamically select a state using the dropdown filter.

- 💡 Insight

Highlights the most vulnerable districts in each state
Helps prioritize high-risk regions for intervention
Enables quick comparison across states

In [3]:
# Prepare data
district_risk = df.groupby(['state','district'])['risk_score'].mean().reset_index()

# initial
first_state = district_risk['state'].unique()[0]
init_df = district_risk[district_risk['state'] == first_state].nlargest(10, 'risk_score')

fig = go.Figure()

fig.add_trace(go.Bar(
    x=init_df['district'],
    y=init_df['risk_score'],
    marker=dict(color=init_df['risk_score'], colorscale='Tealgrn')
))

# slicer
buttons = []
for state in district_risk['state'].unique():
    temp = district_risk[district_risk['state'] == state].nlargest(10, 'risk_score')

    buttons.append(dict(
        label=state,
        method='update',
        args=[{
            'x': [temp['district']],
            'y': [temp['risk_score']],
            'marker.color': [temp['risk_score']]
        }]
    ))

fig.update_layout(
    title='Top 10 Crisis Districts (State Filter)',
    template='plotly_white',
    title_x=0.5,
    updatemenus=[dict(x=0, y=1.15, buttons=buttons)]
)

fig.show()

###  2 Over-Exploited District Count

- Districts with extraction ratio greater than 1 are over-exploited.

In [4]:
df['over_exploited'] = df['extraction_ratio'].apply(
    lambda x: 'Over-Exploited' if x > 1 else 'Normal'
)

count_exploit = df['over_exploited'].value_counts().reset_index()
count_exploit.columns = ['status', 'count']

fig1 = px.bar(
    count_exploit,
    x='status',
    y='count',
    color='status',
    title='Over-Exploited District Count',
    color_discrete_sequence=['#2CA58D', '#0B3C5D']  # ✅ Blue-Green
)

fig1.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig1.show()

###  3 Stress Level vs Critical Zones

Higher stress levels indicate more critical groundwater conditions.

In [5]:
fig2 = px.scatter(
    df,
    x='stress_level',
    y='risk_score',
    color='risk_score',
    size='risk_score',
    hover_name='district',
    title='Stress Level vs Risk',
    color_continuous_scale=px.colors.sequential.Blugrn
)

fig2.update_layout(template='plotly_white', title_x=0.5)
fig2.show()

### 4. Extraction Ratio vs Risk 
- Description

This scatter plot shows the relationship between extraction ratio and risk score, with a state-wise filter.

- 💡 Insight
Districts in the top-right region indicate extreme crisis zones
Higher extraction ratios generally lead to increased risk
Useful for identifying over-exploited groundwater regions

In [6]:
fig = go.Figure()

# -------------------------------
# Initial State
# -------------------------------
first_state = df['state'].unique()[0]
init_df = df[df['state'] == first_state]

# -------------------------------
# Bar Chart
# -------------------------------
fig.add_trace(go.Bar(
    x=init_df['district'],
    y=init_df['extraction_ratio'],
    
    marker=dict(
        color=init_df['risk_score'],   # color = risk
        colorscale='Tealgrn',
        colorbar=dict(title="Risk Score")
    ),

    customdata=init_df[['risk_score']].values,

    hovertemplate=
        "<b>%{x}</b><br>" +
        "Extraction Ratio: %{y:.2f}<br>" +
        "Risk Score: %{customdata[0]:.2f}<extra></extra>"
))

# -------------------------------
# SLICER (Dropdown)
# -------------------------------
buttons = []

for state in df['state'].unique():
    temp = df[df['state'] == state]

    buttons.append(dict(
        label=state,
        method='update',
        args=[{
            'x': [temp['district']],
            'y': [temp['extraction_ratio']],
            'marker.color': [temp['risk_score']],
            'customdata': [temp[['risk_score']].values]
        }]
    ))

# -------------------------------
# Layout
# -------------------------------
fig.update_layout(
    title='Extraction Ratio & Risk (State Filter)',
    template='plotly_white',
    title_x=0.5,
    xaxis_title='District',
    yaxis_title='Extraction Ratio',
    updatemenus=[dict(
        x=0,
        y=1.15,
        buttons=buttons
    )]
)

# -------------------------------
# Show
# -------------------------------
fig.show()

### 5 High Risk Threshold Alert

- Highlights districts crossing a critical risk threshold.

In [7]:
threshold = df['risk_score'].mean()

df['alert'] = df['risk_score'].apply(
    lambda x: 'Critical' if x > threshold else 'Safe'
)

alert_count = df['alert'].value_counts().reset_index()
alert_count.columns = ['alert', 'count']

fig3 = px.pie(
    alert_count,
    names='alert',
    values='count',
    title='High Risk Alert Zones',
    hole=0.5,  # 🔥 donut style
    color='alert',
    color_discrete_sequence=['#2CA58D', '#0B3C5D']  # Blue-Green
)

fig3.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig3.show()

### 6 Extraction Ratio vs Risk Score
- 🔍 Description

This scatter plot visualizes the relationship between extraction ratio and risk score across different districts. Each point represents a district, and the color intensity reflects the risk level.

In [8]:
fig4 = px.scatter(
    df,
    x='extraction_ratio',
    y='risk_score',
    color='risk_score',
    hover_name='district',
    title='Extraction Ratio vs Risk',
    color_continuous_scale=px.colors.sequential.Blugrn_r
)

fig4.update_layout(template='plotly_white', title_x=0.5)
fig4.show()

### 7  Alert Distribution

- Shows proportion of safe vs critical districts.

In [9]:
fig5 = px.pie(
    df,
    names='alert',
    title='Alert Distribution',
    color_discrete_sequence=px.colors.sequential.Blugrn_r  # ✅ Blue-Green
)

fig5.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig5.show()

### 8 Top Crisis Districts

- Districts with highest risk and stress levels.

In [10]:
top_crisis = df.sort_values('risk_score', ascending=False).head(10)

fig6 = px.bar(
    top_crisis,
    x='district',
    y='risk_score',
    color='risk_score',
    title='Top Crisis Districts',
    color_continuous_scale=px.colors.sequential.Blugrn
)

fig6.update_layout(template='plotly_white', title_x=0.5)
fig6.show()

### 9 Risk Level Segmentation

- Divides districts into Low, Medium, and High risk levels.

In [11]:
df['risk_level'] = pd.cut(
    df['risk_score'],
    bins=3,
    labels=['Low', 'Medium', 'High']
)

risk_seg = df['risk_level'].value_counts().reset_index()
risk_seg.columns = ['risk_level', 'count']

fig7 = px.bar(
    risk_seg,
    x='risk_level',
    y='count',
    color='risk_level',
    title='Risk Level Segmentation',
    color_discrete_sequence=['#2CA58D', '#1F7A8C', '#0B3C5D']  # Blue-Green
)

fig7.update_layout(
    template='plotly_white',
    title_x=0.5,
    xaxis_title='Risk Level',
    yaxis_title='Number of Districts'
)

fig7.show()

### 10 Risk Threshold Breach Analysis
📌 Description

- This chart identifies the number of districts that exceed a critical risk threshold (average risk score).

- 💡 Insight

Districts above the threshold are marked as high-alert zones
Provides a clear view of risk distribution across regions
Supports early warning and monitoring systems

In [12]:
threshold = df['risk_score'].mean()

df['alert_flag'] = df['risk_score'] > threshold

alert_df = df.groupby('alert_flag').size().reset_index(name='count')

fig = px.bar(
    alert_df,
    x='alert_flag',
    y='count',
    color='count',
    color_continuous_scale='Tealgrn',
    title='Risk Threshold Breach'
)

fig.update_layout(template='plotly_white', title_x=0.5)

fig.show()

### 11 Stress vs Risk Clustering
- 📌 Description

This scatter plot visualizes the relationship between stress level and risk score across districts.

- 💡 Insight

Helps identify clusters of safe, moderate, and critical zones
High stress + high risk indicates severe groundwater crisis
Useful for segmentation and policy planning-

In [13]:
fig = px.scatter(
    df,
    x='stress_level',
    y='risk_score',
    color='risk_score',
    color_continuous_scale='Tealgrn',
    title='Stress vs Risk Clustering',
    hover_name='district'
)

fig.update_layout(template='plotly_white', title_x=0.5)

fig.show()

### 12.📊 Interactive Crisis Detection Table
- 📌 Multi-Filter Groundwater Risk Analysis

- This interactive table provides a comprehensive view of groundwater crisis indicators across districts. It allows users to dynamically filter data based on state and risk level, making it a powerful tool for detailed analysis.

### 13. State → District Groundwater Risk Treemap'

In [14]:
# Create clean dataset (avoid clutter)
tree_df = df.groupby(['state','district']).agg({
    'extraction_ratio': 'mean',
    'risk_score': 'mean'
}).reset_index()

fig = px.treemap(
    tree_df,
    path=['state', 'district'],   # hierarchy
    values='extraction_ratio',   # size
    color='risk_score',          # color
    color_continuous_scale='Tealgrn',
    title='State → District Groundwater Risk Treemap',
    hover_data={
        'extraction_ratio': ':.2f',
        'risk_score': ':.2f'
    }
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig.show()

### 🔍 Key Takeaways

- Over-exploitation is a major cause of groundwater crisis  
- High stress areas need immediate attention  
- Risk thresholds help in early warning systems  
- Monitoring critical districts can prevent future water shortages  